# HierarchicalDet Phase 0 setup (Kaggle GPU notebook)

Scope: environment install, DENTEX dataset download + 3-tier verification, and a single-image forward-pass smoke test of the inference pipeline (no pretrained HierarchicalDet weights are publicly released, so this only proves the pipeline runs mechanically -- Phase 2 covers actual training/eval).

**Before running**: Settings > Accelerator > GPU T4x2 (or P100). Settings > Internet > On. Settings > Persistence > "Variables and files" (keeps the cloned repo / downloaded dataset across sessions -- the 11.8GB DENTEX download is slow, no reason to repeat it). Settings > Environment > "Pin to original environment".

**Every cell in this notebook is safe to re-run, and the whole notebook is safe to run top-to-bottom with "Run All" even after a session restart.** Kaggle's "pin to original environment" reverts to the base image's default packages on every fresh session -- anything pip-installed outside `/kaggle/working` does NOT survive a restart, only files under `/kaggle/working` do. So dependency installation is written to always re-run quickly (pip no-ops on already-satisfied packages) rather than being a one-time step, and the dataset download/extraction steps check for existing files first and skip the slow parts if already done.

If you ever hit a `PIL.Image` `AttributeError` that this notebook's self-healing step doesn't resolve, use Kaggle's Restart Session button, then Run All from the top again -- see the note in the dependency-install cell for why.

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU in notebook Settings before proceeding.'

In [ ]:
# Clone (or update) the repo. Idempotent: safe to re-run after a restart --
# pulls latest instead of re-cloning if /kaggle/working/repo already exists.
REPO_URL = 'https://github.com/christopherh-88/HierarchicalDet.git'
%cd /kaggle/working
import os
if os.path.isdir('repo/.git'):
    %cd repo
    !git pull
else:
    !git clone $REPO_URL repo
    %cd repo

In [ ]:
# Install dependencies. Always re-run this on a fresh session -- see the
# top markdown cell for why pip installs don't persist across restarts here.
#
# Pillow needs special handling: this old detectron2 snapshot uses
# PIL.Image.LINEAR, an attribute Pillow 10 removed. Kaggle's base image ships
# Pillow >= 10 (confirmed by a scikit-image dependency conflict warning during
# install), and there are two distinct ways that can still bite even after
# installing 'pillow<10': (a) a fresh session simply hasn't had the pin
# applied yet, or (b) something (e.g. matplotlib's backend init) imported PIL
# into this kernel process before this cell ran, so the already-loaded module
# object is stale even though the correct version is now on disk. This cell
# handles both: force-reinstall the pin, then importlib.reload() to pick up
# the new version in the current process, then hard-fail with a clear message
# (rather than a cryptic AttributeError deep in a detectron2 import) if reload
# alone couldn't fix it -- that residual case needs an actual kernel restart.
!grep -v -E '^(torch|torchvision|torchaudio)$' requirements.txt > /tmp/reqs_no_torch.txt
!pip install -q -r /tmp/reqs_no_torch.txt
!pip install -q --force-reinstall --no-deps 'pillow<10'

import importlib
import PIL
importlib.reload(PIL)
import PIL.Image
importlib.reload(PIL.Image)

print('Pillow version:', PIL.__version__)
if not hasattr(PIL.Image, 'LINEAR'):
    raise RuntimeError(
        'PIL.Image.LINEAR still missing after reinstall + reload. '
        'Use Kaggle\'s Restart Session button, then Run All from the top -- '
        'something pre-imported a newer PIL before this cell ran in a way '
        'a plain reload could not undo.'
    )
print('PIL.Image.LINEAR present -- OK')

In [ ]:
# Bundled pycocotools/ (custom categories_1/2/3 3-tier format) ships without
# its compiled _mask Cython extension, and repo-root imports always shadow
# the pip-installed pycocotools package. Copy the compiled extension for this
# interpreter/platform from the pip install into the bundled package.
import site, glob, shutil
site_pkgs = site.getsitepackages()[0]
mask_so = glob.glob(f'{site_pkgs}/pycocotools/_mask*.so')
assert mask_so, f'no compiled _mask extension found under {site_pkgs}/pycocotools'
shutil.copy(mask_so[0], 'pycocotools/')
print('copied', mask_so[0])

In [ ]:
# Verify the full import chain, same checks as local setup.sh
import detectron2
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from hierarchialdet import add_diffusiondet_config, DiffusionDetWithTTA
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper
from pycocotools.coco import COCO
import evaluator
print('All imports OK.')

## Download DENTEX and verify all three annotation tiers

Confirmed via the live HuggingFace file listing (2026-07-14): the dataset repo
contains `DENTEX/training_data.zip`, `DENTEX/validation_data.zip`,
`DENTEX/test_data.zip`, and a standalone `DENTEX/validation_triple.json` (the
fully 3-tier-annotated validation set). The internal structure of the zips
isn't documented anywhere (checked the HF dataset card, its README, and the
DENTEX GitHub README -- none specify it), so the cells below extract
everything and auto-discover the tier files by filename keyword rather than
assuming a fixed path.

In [ ]:
# Download (skips already-downloaded files automatically -- huggingface_hub
# checks local_dir contents) and extract (skips already-extracted zips via a
# marker file, since zipfile has no built-in idempotency check).
import os, zipfile
from huggingface_hub import snapshot_download

RAW_DIR = '/kaggle/working/dentex_raw'
EXTRACT_DIR = '/kaggle/working/dentex_extracted'
os.makedirs(EXTRACT_DIR, exist_ok=True)

dataset_dir = snapshot_download(
    repo_id='ibrahimhamamci/DENTEX',
    repo_type='dataset',
    local_dir=RAW_DIR,
)
print('downloaded to', dataset_dir)

for zip_rel in ['DENTEX/training_data.zip', 'DENTEX/validation_data.zip', 'DENTEX/test_data.zip']:
    zip_path = os.path.join(dataset_dir, zip_rel)
    marker = os.path.join(EXTRACT_DIR, '.' + zip_rel.replace('/', '_') + '.extracted')
    if os.path.exists(marker):
        print('already extracted:', zip_rel)
        continue
    if not os.path.exists(zip_path):
        print('MISSING (not downloaded?):', zip_path)
        continue
    print('extracting', zip_path, '...')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(EXTRACT_DIR)
    open(marker, 'w').close()
    print('done:', zip_rel)

In [ ]:
# Print the extracted tree (depth-limited) and every JSON file found, so you
# can sanity check the auto-discovery in the next cell against ground truth.
print('=== directory tree under', EXTRACT_DIR, '(depth <= 3) ===')
for root, dirs, files in os.walk(EXTRACT_DIR):
    depth = root[len(EXTRACT_DIR):].count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root) or root}/')
    for f in sorted(files)[:20]:
        print(f'{indent}  {f}')
    if len(files) > 20:
        print(f'{indent}  ... ({len(files) - 20} more files)')

print()
print('=== every .json file found (extracted dirs + top-level dataset dir) ===')
json_files = sorted(set(
    os.path.join(root, f)
    for search_root in (EXTRACT_DIR, dataset_dir)
    for root, _, files in os.walk(search_root)
    for f in files if f.endswith('.json')
))
for jf in json_files:
    print(jf)

In [ ]:
# Auto-match the three tiers by filename keyword. This is a best-effort
# heuristic since the zip internals aren't documented -- if it finds nothing
# or picks the wrong file for a tier, use the full listing printed above to
# fix tier_paths by hand.
import json

def find_best_match(files, must_include, exclude=()):
    return [
        f for f in files
        if all(k in f.lower() for k in must_include)
        and not any(e in f.lower() for e in exclude)
    ]

tier_paths = {}

# validation_triple.json is a known, confirmed top-level file (per the HF file
# listing) containing the fully 3-tier-annotated validation set -- this is
# the file the earlier project audit flagged as the primary mAP eval set.
triple_matches = [f for f in json_files if os.path.basename(f) == 'validation_triple.json']
if triple_matches:
    tier_paths['quadrant-enumeration-diagnosis (validation_triple, confirmed)'] = triple_matches[0]

for tier, (must, exclude) in {
    'quadrant': (['quadrant'], ['enumeration', 'disease', 'diagnosis']),
    'quadrant-enumeration': (['enumeration'], ['disease', 'diagnosis']),
    'quadrant-enumeration-diagnosis': (['diagnosis'], []),
}.items():
    matches = find_best_match(json_files, must, exclude)
    if matches:
        tier_paths.setdefault(tier, matches[0])
        if len(matches) > 1:
            print(f'NOTE: multiple candidates matched "{tier}", picked the first: {matches}')

if not tier_paths:
    print('No tier files auto-matched at all -- inspect the tree/JSON listing in the previous cell and set tier_paths manually.')

for tier, path in tier_paths.items():
    try:
        d = json.load(open(path))
        print(tier, '->', path)
        print('  images:', len(d.get('images', [])), 'annotations:', len(d.get('annotations', [])))
        print('  category keys:', [k for k in d if k.startswith('categor')])
    except Exception as e:
        print(tier, '->', path, 'FAILED TO LOAD:', e)

## Single-image forward-pass smoke test

No pretrained HierarchicalDet weights exist publicly, so this uses the public
Swin-B ImageNet-22k backbone per `configs/diffdet.custom.swinbase.nonpretrain.yaml`,
with the detection head randomly initialized. The goal is only to prove the
model builds and runs a forward pass on GPU without crashing -- not to
produce meaningful detections. The test image is auto-picked from the
extracted dataset rather than needing a manually typed path.

In [ ]:
# Download the Swin-B backbone (skipped if already present).
# IMPORTANT: keep the .pth extension -- detectron2's DetectionCheckpointer
# dispatches purely on file extension (detectron2/checkpoint/detection_checkpoint.py
# :_load_file): ".pkl" is parsed as a Caffe2-style pickle blob, which would
# silently mis-load a native torch.save() file. A non-".pkl"/".pyth" extension
# routes through the normal torch.load() + name-matching-heuristics path,
# which is what this raw Swin-Transformer release checkpoint needs.
os.makedirs('models', exist_ok=True)
WEIGHTS_PATH = 'models/swin_base_patch4_window7_224_22k.pth'
if not os.path.exists(WEIGHTS_PATH):
    !wget -q -O $WEIGHTS_PATH https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_base_patch4_window7_224_22k.pth
print(WEIGHTS_PATH, os.path.getsize(WEIGHTS_PATH), 'bytes')
# NOTE: detectron2 loads this via name-matching heuristics
# (align_and_update_state_dicts), which only works if this checkpoint's
# state_dict keys line up with hierarchialdet/swintransformer.py's module
# names -- not yet confirmed. Watch for "parameters/buffers not found"
# warnings in the next cell's output; if most/all keys mismatch, the backbone
# is effectively randomly initialized and a key-renaming step will be needed.

In [ ]:
# Auto-pick one real image from the extracted dataset and run demo.py on it.
import glob

candidate_images = []
for ext in ('*.png', '*.jpg', '*.jpeg'):
    candidate_images += glob.glob(os.path.join(EXTRACT_DIR, '**', ext), recursive=True)

assert candidate_images, f'No images found under {EXTRACT_DIR} -- check the extraction cell above ran successfully.'
sample_image = sorted(candidate_images)[0]
print('using sample image:', sample_image)

!python demo.py \
  --config-file configs/diffdet.custom.swinbase.nonpretrain.yaml \
  --input "$sample_image" \
  --output /kaggle/working/smoke_test_output.jpg \
  --opts MODEL.WEIGHTS $WEIGHTS_PATH MODEL.DEVICE cuda